# 01 · Pengumpulan Data
**Bab III Subbab 3.2.** Mengunduh bagian *User Reviews* Amazon Reviews 2023 untuk kategori yang dipilih kelompok,
lalu mencatat URL, tanggal unduh, ukuran berkas, jumlah baris, dan checksum.

**Keluaran:** `data/raw/<kategori>.jsonl`, `outputs/<kategori>/manifest_raw.json`, `outputs/<kategori>/config.json`

> **Google Colab.** Jalankan sekali per sesi sebelum sel setup:
> ```
> !git clone -b main https://github.com/Vito28/dl-paper.git
> %cd dl-paper/notebooks
> !pip install -q -r ../requirements.txt
> ```
> Sel setup otomatis me-mount Google Drive (izinkan akses saat diminta). **Semua data dan hasil, termasuk
> unduhan mentah, disimpan di `MyDrive/dl-paper`**, bukan di instance. Jika Drive gagal di-mount atau
> terputus, notebook berhenti dengan error. Jalankan sel terakhir (`finish()`) agar semua tulisan tersinkron.

In [ ]:
# Setup: path proyek dan konfigurasi bersama (ubah konfigurasi di notebooks/settings.py)
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "notebooks")]
from settings import cfg, SMOKE, finish
import numpy as np, pandas as pd
pd.set_option("display.width", 160)
print("kategori:", cfg.category, "| output:", cfg.out_dir, "| SMOKE:", SMOKE)

## 1. Simpan konfigurasi yang dipakai
Konfigurasi disimpan agar setiap hasil di Bab IV bisa ditelusuri ke pengaturan yang menghasilkannya.

In [ ]:
print(cfg.save())
print("eksperimen yang akan dijalankan:", cfg.experiments_to_run(), cfg.scales_to_run(), cfg.targets_to_run())

## 2. Unduh berkas User Reviews
Berkas diambil dari repositori resmi McAuley Lab di Hugging Face. Ukuran berkas bisa beberapa GB, jadi unduhan hanya dilakukan sekali.

In [ ]:
from hr import data
if SMOKE:
    from hr.synthetic import synthetic
    cfg.raw_path.parent.mkdir(parents=True, exist_ok=True)
    synthetic(n_products=600).to_json(cfg.raw_path, orient="records", lines=True)
manifest = data.download_raw(cfg)
manifest

## 3. Cek skema
Pastikan semua kolom yang dibutuhkan ada dan lihat contoh nilainya, terutama satuan `timestamp`.

In [ ]:
from hr.config import RAW_COLUMNS
with open(cfg.raw_path) as f:
    sample = [json.loads(next(f)) for _ in range(5)]
missing = [c for c in RAW_COLUMNS if c not in sample[0]]
assert not missing, f"Kolom wajib tidak ada di berkas mentah: {missing}"
pd.DataFrame(sample)[RAW_COLUMNS]

## Simpan ke Google Drive
Memastikan semua hasil notebook ini sudah tersinkron ke Drive sebelum runtime ditutup.

In [ ]:
finish()